# 02 — Hai baseline chưa huấn luyện (zero-shot)

> Notebook trình bày, **không huấn luyện lại**. Mọi bảng và hình được tính từ file kết quả đã lưu trong `saved/`, chạy lại an toàn trên CPU. Log huấn luyện đã được rút gọn (bỏ thanh tiến độ, giữ mọi kết quả eval); bản đầy đủ nằm trong `ASR/notebooks/local` và `ASR/local/logs`.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
S = next(p / 'saved' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'saved/tables').is_dir())
T = S / 'tables'
def js(rel): return json.loads((S / rel).read_text(encoding='utf-8'))
def show_log(name): print((S / 'logs' / name).read_text(encoding='utf-8'))
def history(name): return js(f'runs/{name}_trainer_state.json')['log_history']
pd.set_option('display.precision', 2)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
C = {'Baseline 1': '#7E9C9B', 'Baseline 2': '#A9C2C0', 'V1': '#0F4E57', 'V2': '#1A6771', 'V-Final': '#FFB21A'}

Cùng dữ liệu, cùng luật chuẩn hóa, cùng giải mã greedy. Baseline không huấn luyện nên không có lr, optimizer, epoch.
Phần cứng: RTX 4050 Laptop 6 GB; PyTorch 2.6.0, Transformers 4.46.3.

In [2]:
print((S / 'configs/_base.yaml').read_text(encoding='utf-8'))

# Shared defaults. Each experiments/<ID>/config.yaml only overrides what differs.
seed: 13

model:
  name: vinai/PhoWhisper-small
  language: vi
  task: transcribe
  lora: null                 # null = no LoRA (zero-shot, or full fine-tune if `train` is set)

data:
  dir: data                  # relative to ASR/ ; override with --data-dir on Colab
  normalize_text: true       # lowercase + strip punctuation for train targets (same rule as scoring)
  max_duration: 30.0         # Whisper window
  train_sources:             # source: max_hours (null = all of that source's train split)
    VIVOS: null
    FLEURS: null
    VietMed: null
    ViMedCSS: null
  eval_sources: [VIVOS, FLEURS, VietMed, ViMedCSS]
  dev_max_per_source: 150    # in-training dev subset (keeps eval cheap); full dev via evaluate.py

train: null                  # null = evaluation-only experiment (baseline)

eval:
  batch_size: 16
  num_beams: 1
  max_new_tokens: 225



**Lệnh đã chạy — Baseline 1 trên toàn bộ DEV** (chép từ `ASR/notebooks/local/01_B0.ipynb`, ô 5; chỉ để xem)

```python
print('FULL DEV EVALUATION: all DEV clips', flush=True)
run_cli([RUNNER, 'eval', EXP, '--run-id', 'dev_local_v1', *eval_options])
```

In [3]:
show_log('Baseline1_dev_eval.txt')

===== 20261018_002135_406032.log
[INIT] mode=eval experiment=B0_phowhisper_small_zeroshot; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  …
}
[MODEL] Loading processor and weights; first use may download model files...
[MODEL] Weights loaded.
Exclude ViMedCSS/dev: 1 empty-audio rows (duration <= 0): ['vimedcss:Med_CS-4072_1-11']
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
5168/5168
{
  "overall_wer": 0.2472667026156816,
  "by_source_wer": {
    "FLEURS": 0.10885805763073639,
    "VIVOS": 0.0042037586547972305,
    "ViMedCSS": 0.28362952683647435,
    "VietMed": 0.

In [4]:
show_log('Baseline2_dev_eval.txt')

===== 20261018_024114_120391.log
[INIT] mode=eval experiment=B1_whisper_small_zeroshot; importing PyTorch/ASR libraries...
{
  "python": "D:\\1.FPT_FALL26\\DPL - Deep Learning\\.venv-asr\\Scripts\\python.exe",
  "cuda": true,
  "versions": {
    "torch": "2.6.0+cu124",
    "transformers": "4.46.3",
    "peft": "0.13.2",
    "accelerate": "1.1.1",
    "datasets": "3.1.0"
  },
  …
}
[MODEL] Loading processor and weights; first use may download model files...
[MODEL] Weights loaded.
Exclude ViMedCSS/dev: 1 empty-audio rows (duration <= 0): ['vimedcss:Med_CS-4072_1-11']
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
5168/5168
{
  "overall_wer": 0.5875163138192703,
  "by_source_wer": {
    "FLEURS": 0.256136606189968,
    "VIVOS": 0.20450049455984173,
    "ViMedCSS": 0.4645463091454407,
    "VietMed": 0.73924270

In [5]:
def dev_row(name):
    m = js(f'metrics/{name}.json'); o, s = m['overall'], m['by_source']
    return {'WER %': 100 * o['wer'], 'CER %': 100 * o['cer'], 'Recall thuật ngữ %': 100 * o.get('cs_term_recall', float('nan')),
            **{k: 100 * s[k]['wer'] for k in ['ViMedCSS', 'VietMed', 'FLEURS', 'VIVOS']}}
DEV = pd.DataFrame({'Baseline 1': dev_row('Baseline1_dev'), 'Baseline 2': dev_row('Baseline2_dev'), }).T
DEV

,WER %,CER %,Recall thuật ngữ %,ViMedCSS,VietMed,FLEURS,VIVOS
Baseline 1,24.73,19.25,12.96,28.36,25.87,10.89,0.42
Baseline 2,58.75,46.35,16.05,46.45,73.92,25.61,20.45


## So với số công bố (paper ViMedCSS), WER %

In [6]:
# ViMedCSS TEST (1.613 clip) và HARD (658 clip); B2, B3 là PhoWhisper-large, Whisper-large-v3 chưa huấn luyện, chạy thêm để kiểm tra cách chấm
g = pd.read_csv(T / 'same_metric_official.csv').query("system in ['B0', 'B2', 'B3']")
p = pd.read_csv(T / 'same_metric_paper.csv').set_index(['split', 'system'])['wer']
g = g.assign(paper=[p.get((r.split, r.paper_row)) for r in g.itertuples()])[['split', 'paper_row', 'paper', 'wer']]
g = g.assign(lech=g.paper - g.wer).rename(columns={'paper_row': 'mô hình', 'wer': 'nhóm', 'lech': 'lệch'}).reset_index(drop=True)
g.loc[len(g)] = ['TEST', 'Whisper-Small zero-shot (Baseline 2)', p[('TEST', 'Whisper-Small zero-shot')], float('nan'), float('nan')]
g

,split,mô hình,paper,nhóm,lệch
0,TEST,PhoWhisper-Small zero-shot,36.31,28.02,8.29
1,TEST,PhoWhisper-Large zero-shot,31.24,23.14,8.10
2,TEST,Whisper-Large-v3 zero-shot,34.47,29.61,4.86
3,HARD,PhoWhisper-Small zero-shot,46.27,36.04,10.23
4,HARD,PhoWhisper-Large zero-shot,37.37,27.14,10.23
5,HARD,Whisper-Large-v3 zero-shot,39.35,33.14,6.21
6,TEST,Whisper-Small zero-shot (Baseline 2),50.03,NaN,NaN


**Gap analysis.** Cùng mô hình, cùng tập, không huấn luyện mà số của nhóm thấp hơn paper 5–10 điểm, nên chênh lệch đến từ
cách chấm (transcript tham chiếu, chuẩn hóa văn bản, scorer), không phải từ siêu tham số. Paper không công bố luật chuẩn hóa.
Baseline 2 chưa được giải mã trên TEST/HARD chính thức, nên chưa có dòng so trực tiếp với paper.